<a href="https://colab.research.google.com/github/WingmanShi/WingmanShi-AnalyticsEngineering/blob/main/Weather_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Task: Extract Weather Data from Open-Meteo

## Goal
Use the Open-Meteo API to download daily weather data for Coventry, UK, and prepare a clean pandas DataFrame. In the next exercise, this DataFrame will be used for calculations, aggregations, and reporting.

**Do not calculate averages, totals, or other summaries in this workbook.**

## Data to retrieve

Location: **Coventry, UK**

- Latitude: `52.4068`
- Longitude: `-1.5197`
- Date range: `2026-01-01` to `2026-01-31`

The API provides daily maximum temperature, minimum temperature, and total precipitation.

## Part 1 — Request the data

Run the cell below. It imports the required libraries, sends the API request, checks that it succeeded, and stores the JSON response in `weather_json`.

In [1]:
import requests
import pandas as pd

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=52.4068&longitude=-1.5197"
    "&start_date=2026-01-01&end_date=2026-01-31"
    "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum"
    "&timezone=Europe%2FLondon"
)

response = requests.get(url)
response.raise_for_status()
weather_json = response.json()

## Part 2 — Inspect the response

Use the cell below to answer these questions:

1. What are the top-level keys in `weather_json`?
2. Which key contains the daily weather observations?
3. What are the keys inside that daily section?

In [2]:
# 1. What are the top-level keys in weather_json?
print("Top-level keys:", list(weather_json.keys()))

Top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']


In [3]:
# Inspect the types of all top-level keys in weather_json
for key, value in weather_json.items():
    print(f"{key}: {type(value)}")

latitude: <class 'float'>
longitude: <class 'float'>
generationtime_ms: <class 'float'>
utc_offset_seconds: <class 'int'>
timezone: <class 'str'>
timezone_abbreviation: <class 'str'>
elevation: <class 'float'>
daily_units: <class 'dict'>
daily: <class 'dict'>


In [4]:
# 2. Which key contains the daily weather observations?
# Confirm that 'daily' is a dictionary containing our daily keys
daily_data = weather_json.get("daily")
print("Contains daily observations key: 'daily'")
print("Is 'daily' a dictionary?", isinstance(daily_data, dict))

if isinstance(daily_data, dict):
    print("Keys inside 'daily':", list(daily_data))

Contains daily observations key: 'daily'
Is 'daily' a dictionary? True
Keys inside 'daily': ['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum']


## Part 3 — Create a DataFrame

Create a DataFrame from the API's daily section. It must contain the following columns, with exactly these names:

- `date`
- `max_temperature_c`
- `min_temperature_c`
- `precipitation_mm`

Hint: the data in the daily section is already arranged as columns of equal-length lists. `pd.DataFrame(...)` can convert it into a DataFrame.

In [6]:
# 1. Create a DataFrame from weather_json["daily"].
df = pd.DataFrame(weather_json["daily"])

# 2. Rename its columns to the required names above.
df = df.rename(columns={
    "time": "date",
    "temperature_2m_max": "max_temperature_c",
    "temperature_2m_min": "min_temperature_c",
    "precipitation_sum": "precipitation_mm"
})




In [7]:
# show the first 5 lines of the datata frame to inspect contents

display(df.head())


,date,max_temperature_c,min_temperature_c,precipitation_mm
0,2026-01-01,5.5,0.8,0.7
1,2026-01-02,2.5,-0.7,0.7
2,2026-01-03,1.4,-2.1,0.0
3,2026-01-04,2.0,-3.4,0.0
4,2026-01-05,0.2,-3.9,0.4


## Part 4 — Clean and inspect the DataFrame

1. Convert the `date` column to a datetime type using pandas.
2. Display the first five rows.
3. Display the DataFrame dimensions.
4. Display the column data types.

Hint: use `pd.to_datetime()`, `.head()`, `.shape`, and `.dtypes`.

In [8]:
# inspect the data tpyes, to see if we need to clean the data

df.dtypes

,0
date,object
max_temperature_c,float64
min_temperature_c,float64
precipitation_mm,float64


In [10]:
# 1. Convert the date column to datetime.
df['date'] = pd.to_datetime(df['date'])

# 2. Display the first five rows.
display(df.head())

# 3. Display the DataFrame dimensions.
print("Dimensions (shape):", df.shape)

# 4. Display the column data types.
print("\nData types:\n", df.dtypes)


,date,max_temperature_c,min_temperature_c,precipitation_mm
0,2026-01-01,5.5,0.8,0.7
1,2026-01-02,2.5,-0.7,0.7
2,2026-01-03,1.4,-2.1,0.0
3,2026-01-04,2.0,-3.4,0.0
4,2026-01-05,0.2,-3.9,0.4


Dimensions (shape): (31, 4)

Data types:
 date                 datetime64[ns]
max_temperature_c           float64
min_temperature_c           float64
precipitation_mm            float64
dtype: object


Task: Calculate the average daily temperature

In [13]:
df['temp_range'] = df['max_temperature_c'] - df['min_temperature_c']


df.head()

,date,max_temperature_c,min_temperature_c,precipitation_mm,temp_range
0,2026-01-01,5.5,0.8,0.7,4.7
1,2026-01-02,2.5,-0.7,0.7,3.2
2,2026-01-03,1.4,-2.1,0.0,3.5
3,2026-01-04,2.0,-3.4,0.0,5.4
4,2026-01-05,0.2,-3.9,0.4,4.1


Questions:

1.   Which date had the largest temperature range?
2.   What was that range, in °C?
3.   Which date had the smallest temperature range?



In [16]:
# 1. Which date had the largest temperature range?
max_row = df.loc[df['temp_range'].idxmax()]
print(f"Day with the largest temperature range: {max_row['date'].strftime('%Y-%m-%d')}, Range: {max_row['temp_range']:.1f} °C")

# 2. Which day has the smallest temperature range?
min_row = df.loc[df['temp_range'].idxmin()]
print(f"Day with the smallest temperature range: {min_row['date'].strftime('%Y-%m-%d')}, Range: {min_row['temp_range']:.1f} °C")



Day with the largest temperature range: 2026-01-11, Range: 11.0 °C
Day with the smallest temperature range: 2026-01-26, Range: 2.2 °C


Calcualte the monthly summary figures.

In [18]:
print("\n--- Monthly Summary ---")
print(f"Mean maximum temperature: {df['max_temperature_c'].mean():.1f} °C")
print(f"Mean minimum temperature: {df['min_temperature_c'].mean():.1f} °C")
print(f"Mean daily temperature range: {df['temp_range'].mean():.1f} °C")
print(f"Total precipitation: {df['precipitation_mm'].sum():.1f} mm")
print(f"Dry days: {(df['precipitation_mm'] == 0).sum()}")
print(f"Wet days: {(df['precipitation_mm'] > 0).sum()}")

wettest = df.loc[df['precipitation_mm'].idxmax()]
print(f"Wettest day: {wettest['date'].strftime('%Y-%m-%d')} ({wettest['precipitation_mm']} mm)")


--- Monthly Summary ---
Mean maximum temperature: 6.5 °C
Mean minimum temperature: 2.0 °C
Mean daily temperature range: 4.5 °C
Total precipitation: 99.8 mm
Dry days: 3
Wet days: 28
Wettest day: 2026-01-08 (16.3 mm)


Aggregate the data by week

In [19]:
# Ensure 'date' is the index and set to datetime if it isn't already
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df_weekly = df.set_index('date')
else:
    df_weekly = df.copy()

# Resample by week ('W') and aggregate:
# - Mean for temperatures and temperature range
# - Sum for precipitation
weekly_summary = df_weekly.resample('W').agg({
    'max_temperature_c': 'mean',
    'min_temperature_c': 'mean',
    'temp_range': 'mean',
    'precipitation_mm': 'sum'
}).round(1)

print("--- Weekly Aggregated Summary ---")
display(weekly_summary)


--- Weekly Aggregated Summary ---


,max_temperature_c,min_temperature_c,temp_range,precipitation_mm
date,,,,
2026-01-04,2.8,-1.4,4.2,1.4
2026-01-11,4.1,-1.2,5.3,33.5
2026-01-18,8.3,3.1,5.1,22.2
2026-01-25,8.6,5.3,3.2,25.0
2026-02-01,7.3,2.7,4.6,17.7


Create a monthly report

In [20]:
# Calculate monthly report metrics
monthly_report = {
    "Total Days": len(df),
    "Mean Max Temp (°C)": round(df['max_temperature_c'].mean(), 1),
    "Mean Min Temp (°C)": round(df['min_temperature_c'].mean(), 1),
    "Mean Daily Range (°C)": round(df['temp_range'].mean(), 1),
    "Total Precipitation (mm)": round(df['precipitation_mm'].sum(), 1),
    "Total Dry Days": int((df['precipitation_mm'] == 0).sum()),
    "Total Wet Days": int((df['precipitation_mm'] > 0).sum())
}

# Convert to a clean DataFrame for display
monthly_report_df = pd.DataFrame(list(monthly_report.items()), columns=["Metric", "Value"])

print("=== Monthly Weather Report ===")
display(monthly_report_df)


=== Monthly Weather Report ===


,Metric,Value
0,Total Days,31.0
1,Mean Max Temp (°C),6.5
2,Mean Min Temp (°C),2.0
3,Mean Daily Range (°C),4.5
4,Total Precipitation (mm),99.8
5,Total Dry Days,3.0
6,Total Wet Days,28.0


Extension Tasks:

1.   Average precipitation by day of the week
2.   Rain categories
3.   Days warmer than the monthly average maximum



In [22]:
import pandas as pd

# 1. Average precipitation by day of the week
# Extract the day name (e.g., 'Monday', 'Tuesday') and group by it
df['day_of_week'] = df['date'].dt.day_name()

# Optional: Ensure days are ordered chronologically instead of alphabetically
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_of_week'] = pd.Categorical(df['day_of_week'], categories=days_order, ordered=True)

avg_precip_by_day = df.groupby('day_of_week', observed=False)['precipitation_mm'].mean().reset_index()
print("--- Average Precipitation by Day of the Week ---")
display(avg_precip_by_day)


# 2. Rain categories
# Define a function to categorize days based on precipitation amount (in mm)
def categorize_rain(precip):
    if precip == 0:
        return 'No Rain'
    elif precip <= 2.5:
        return 'Light Rain'
    elif precip <= 10.0:
        return 'Moderate Rain'
    else:
        return 'Heavy Rain'

df['rain_category'] = df['precipitation_mm'].apply(categorize_rain)

print("\n--- Rain Category Counts ---")
display(df['rain_category'].value_counts())


# 3. Days warmer than the monthly average maximum
# Calculate the monthly average for maximum temperature
monthly_mean_max = df['max_temperature_c'].mean()

# Filter days where max temperature is strictly greater than the monthly average
warmer_days = df[df['max_temperature_c'] > monthly_mean_max]

print(f"\n--- Days Warmer Than Monthly Average Max ({monthly_mean_max:.1f} °C) ---")
print(f"Total count: {len(warmer_days)} days out of {len(df)}")
display(warmer_days[['date', 'max_temperature_c']])

--- Average Precipitation by Day of the Week ---


,day_of_week,precipitation_mm
0,Monday,0.450
1,Tuesday,4.875
2,Wednesday,4.025
3,Thursday,6.440
4,Friday,3.280
5,Saturday,0.900
6,Sunday,2.325



--- Rain Category Counts ---


,count
rain_category,
Light Rain,16
Moderate Rain,9
No Rain,3
Heavy Rain,3



--- Days Warmer Than Monthly Average Max (6.5 °C) ---
Total count: 19 days out of 31


,date,max_temperature_c
10,2026-01-11,10.4
11,2026-01-12,10.5
12,2026-01-13,9.3
13,2026-01-14,6.7
14,2026-01-15,7.1
15,2026-01-16,7.9
16,2026-01-17,8.8
17,2026-01-18,7.5
18,2026-01-19,8.9
19,2026-01-20,9.4
